Goal: Understand time unrolling, hidden-state flow, and gradient issues in Vanilla RNN


In [77]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [78]:
# =====================
# Hyperparameters
# =====================
SEQ_LEN = 25
BATCH_SIZE = 32
HIDDEN_SIZE = 128
LR = 1e-3
EPOCHS = 20


In [79]:
with open("data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Optional: limit size for fast experiments
text = text[:200_000]

chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encoded = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)

print("Dataset size:", len(encoded))
print("Vocab size:", vocab_size)


Dataset size: 200000
Vocab size: 83


In [80]:
def get_batch(encoded, batch_size, seq_len):
    N = encoded.size(0)
    start_idx = torch.randint(0, N - seq_len - 1, (batch_size,))

    X = torch.stack([encoded[i:i+seq_len] for i in start_idx])
    Y = torch.stack([encoded[i+1:i+seq_len+1] for i in start_idx])

    return X, Y


In [81]:
def one_hot(x, vocab_size):
    return torch.zeros(x.size(0), vocab_size).scatter_(1, x.unsqueeze(1), 1)


In [82]:
class VanillaRNNCell(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_xh = nn.Parameter(torch.randn(vocab_size, hidden_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.b_h  = nn.Parameter(torch.zeros(hidden_size))

        self.W_hy = nn.Parameter(torch.randn(hidden_size, vocab_size) * 0.01)
        self.b_y  = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x_t, h_prev):
        x_onehot = one_hot(x_t, self.W_xh.size(0))
        h_t = torch.tanh(x_onehot @ self.W_xh + h_prev @ self.W_hh + self.b_h)
        y_t = h_t @ self.W_hy + self.b_y
        return h_t, y_t


In [83]:
def forward_rnn(cell, X):
    B, T = X.shape
    h = torch.zeros(B, cell.hidden_size)

    logits = []
    for t in range(T):
        h, y_t = cell(X[:, t], h)
        logits.append(y_t)

    return torch.stack(logits, dim=1)  # (B, T, V)


In [84]:
cell = VanillaRNNCell(vocab_size, HIDDEN_SIZE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cell.parameters(), lr=LR)


In [85]:
for epoch in range(EPOCHS):
    epoch_loss = 0

    for _ in range(100):
        X, Y = get_batch(encoded, BATCH_SIZE, SEQ_LEN)

        logits = forward_rnn(cell, X)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            Y.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cell.parameters(), 5)
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss/100:.4f}")


Epoch 1, Loss: 3.4572
Epoch 2, Loss: 3.2784
Epoch 3, Loss: 3.2716
Epoch 4, Loss: 3.2518
Epoch 5, Loss: 3.2421
Epoch 6, Loss: 3.2406
Epoch 7, Loss: 3.1868
Epoch 8, Loss: 3.1290
Epoch 9, Loss: 3.0502
Epoch 10, Loss: 2.9240
Epoch 11, Loss: 2.7858
Epoch 12, Loss: 2.6727
Epoch 13, Loss: 2.5950
Epoch 14, Loss: 2.5385
Epoch 15, Loss: 2.4651
Epoch 16, Loss: 2.4103
Epoch 17, Loss: 2.3579
Epoch 18, Loss: 2.3131
Epoch 19, Loss: 2.2669
Epoch 20, Loss: 2.2327


In [86]:
def generate_text(cell, start_char, length=300, temperature=1.0):
    cell.eval()

    idx = torch.tensor([char_to_idx[start_char]])
    h = torch.zeros(1, cell.hidden_size)

    generated = [start_char]

    for _ in range(length):
        h, logits = cell(idx, h)
        logits = logits / temperature

        probs = F.softmax(logits.squeeze(0), dim=-1)
        idx = torch.multinomial(probs, 1)

        generated.append(idx_to_char[idx.item()])

    return "".join(generated)


In [87]:
print(generate_text(cell, start_char="T", length=500))


The wind domy,
r whad thou ’ald wamesed’n myoutfiManelitg’nt ny des cid’strteovy coulls canl ton;
Filo efot on from, comt,
 oun cis auln:
Cnet vetle wott rrout,
Butoss cous doteat autheen bepling maungy seng’t; yuse,
No ant!

PEWUIMise tham
Hinith; and sult,
CSan’veant
Ang be omegsing.
Wiir thot rol lott doth here to home worenust en then insitHrer tots in tous, nomlat’cl iy Mokd tr these wendd
Warmer ang  oicnt_
Mathey  yos areste dave hyorr’ang, weld
gr thinde sver?

 Caras efa!  o muveshit dab
